## Install Libraries

In [52]:
pip install streamlit langchain langchain-community langchain-google-genai langchain-groq langchain-text-splitters faiss-cpu pymupdf python-dotenv langsmith jupyter ipykernel sentence-transformers langchain-huggingface

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Importing lib

In [53]:
import os
from dotenv import load_dotenv

# Loader
from langchain_community.document_loaders import PyMuPDFLoader

# Chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings
from langchain_huggingface import HuggingFaceEmbeddings

1
# Vector Store
from langchain_community.vectorstores import FAISS

# LLM
from langchain_groq import ChatGroq

# Prompting
from langchain_core.prompts import ChatPromptTemplate

# Output Parser
from langchain_core.output_parsers import StrOutputParser

print("✅ Imports loaded")

✅ Imports loaded


In [54]:
from dotenv import load_dotenv
import os

load_dotenv(".env", override=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
LANGCHAIN_API_KEY = os.getenv("LANGCHAIN_API_KEY")

print("✅ All API Keys Loaded")

✅ All API Keys Loaded


In [ ]:
CONFIG = {
    "docs_path": "docs/zyro-dynamics-hr-corpus",
    "chunk_size": 700,
    "chunk_overlap": 100,
    "k": 5,
    "fetch_k": 50,
    "similarity_threshold": 1.5,
    "embedding_model": "BAAI/bge-large-en-v1.5",
    "llm_model": "llama-3.3-70b-versatile"
}

print("✅ Config loaded")g




✅ Config loaded


## load Doc

In [56]:
from pathlib import Path

def load_docs(path=CONFIG["docs_path"]):
    documents = []

    for pdf_file in Path(path).glob("*.pdf"):
        loader = PyMuPDFLoader(str(pdf_file))
        docs = loader.load()

        for doc in docs:
            doc.metadata["source_file"] = pdf_file.name

        documents.extend(docs)

    print(f"✅ Loaded {len(documents)} pages")
    return documents

documents = load_docs()

✅ Loaded 39 pages


In [57]:
print(documents[0].metadata)
print(documents[0].page_content[:500])

{'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-05-20T11:14:29+00:00', 'source': 'docs\\zyro-dynamics-hr-corpus\\00_Company_Profile.pdf', 'file_path': 'docs\\zyro-dynamics-hr-corpus\\00_Company_Profile.pdf', 'total_pages': 4, 'format': 'PDF 1.4', 'title': '(anonymous)', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2026-05-20T11:14:29+00:00', 'trapped': '', 'modDate': "D:20260520111429+00'00'", 'creationDate': "D:20260520111429+00'00'", 'page': 0, 'source_file': '00_Company_Profile.pdf'}
Zyro Dynamics Pvt. Ltd.
Company Profile
Doc Code: ZDL-CORP-001
Confidential — For Internal Use Only
Page 1
Zyro Dynamics Pvt. Ltd.
Navigate the Future
Company Profile
Document Code
ZDL-CORP-001
Version
V.01
Effective Date
01 April 2025
Document Owner
Corporate Communications
COMPANY OVERVIEW
Zyro Dynamics Pvt. Ltd. is a product-based enterprise software company headquartered in the United States. 
Founded in 2014 

## Chunking

In [58]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CONFIG["chunk_size"],
    chunk_overlap=CONFIG["chunk_overlap"]
)

chunks = splitter.split_documents(documents)

for i, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = i

print(f"✅ Created {len(chunks)} chunks")

✅ Created 121 chunks


## Embedding

In [59]:

embeddings = HuggingFaceEmbeddings(
    model_name=CONFIG["embedding_model"]
)

print("✅ BGE Large Embeddings Loaded")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

✅ BGE Large Embeddings Loaded


## Vector DB Store

In [60]:
vector = embeddings.embed_query("Who founded Zyro Dynamics?")

print("Dimension:", len(vector))

Dimension: 1024


In [61]:
vector = embeddings.embed_query("Who founded Zyro Dynamics?")

print("Dimension:", len(vector))
print(vector[:5])

Dimension: 1024
[-0.01224010530859232, 0.022082824259996414, -0.025224927812814713, -0.003133386140689254, -0.012426735833287239]


In [62]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    chunks,
    embeddings
)

print("✅ FAISS index created")

✅ FAISS index created


In [63]:
vectorstore.save_local("zyro_hr_faiss")

print("✅ FAISS saved")

✅ FAISS saved


In [64]:
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": CONFIG["k"],
        "fetch_k": CONFIG["fetch_k"]
    }
)

print("✅ MMR Retriever Ready")

✅ MMR Retriever Ready


In [ ]:
def is_in_scope(query):

    result = vectorstore.similarity_search_with_score(
        query,
        k=1
    )

    doc, score = result[0]

    print("Similarity Score:", score)

    return score < CONFIG["similarity_threshold"]

In [66]:
question = "Who founded Zyro Dynamics?"

if is_in_scope(question):
    response = rag_chain.invoke(question)
else:
    response = "I could not find that information in the provided documents."

print(response)

Similarity Score: 0.46978253
1. Lamine Yamal and Aitana Bonmati founded Zyro Dynamics.
2. Zyro Dynamics Pvt. Ltd. Company Profile, Doc Code: ZDL-CORP-001, Page 1.


In [67]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model=CONFIG["llm_model"],
    temperature=0
)

print("✅ Groq LLM Loaded")

✅ Groq LLM Loaded


In [68]:
prompt = ChatPromptTemplate.from_template("""
You are a production-grade RAG assistant.

You must answer questions ONLY from the supplied context.

Instructions:
- Never use external knowledge.
- Never hallucinate information.
- If the answer is not explicitly present in the context, return:

I could not find that information in the provided documents.

- Prefer exact facts from the documents.
- Keep answers clear and concise.
- Cite the source file(s) used at the end.

Context:
{context}

Question:
{question}

Provide:
1. Answer
2. Source(s)
""")

In [70]:
def get_sources(docs):

    sources = []

    for doc in docs:

        source = (
            f"{doc.metadata['source_file']} "
            f"(Page {doc.metadata['page'] + 1})"
        )

        sources.append(source)

    return list(set(sources))

In [71]:
def ask_rag(question):

    if not is_in_scope(question):
        return (
            "I could not find that information "
            "in the provided documents."
        )

    docs = retriever.invoke(question)

    answer = rag_chain.invoke(question)

    sources = get_sources(docs)

    return (
        f"{answer}\n\n"
        f"Sources:\n- "
        + "\n- ".join(sources)
    )

In [72]:
print(
    ask_rag(
        "Who founded Zyro Dynamics?"
    )
)

Similarity Score: 0.46978253
1. Lamine Yamal and Aitana Bonmati founded Zyro Dynamics.
2. Zyro Dynamics Pvt. Ltd. Company Profile, Doc Code: ZDL-CORP-001, Page 1.

Sources:
- 00_Company_Profile.pdf (Page 1)
- 09_Onboarding_and_Separation_Policy.pdf (Page 3)
- 04_Code_of_Conduct.pdf (Page 1)
- 08_Prevention_of_Sexual_Harassment_Policy.pdf (Page 1)


In [ ]:
test_questions = [
    "Who founded Zyro Dynamics?",
    "Where is the headquarters located?",
    "Who is the CEO?",
    "When was the company founded?",
    "What is the employee grade structure?",
    "Which office is in Dubai?",
    "How many employees work in Austin?",
    "What is the capital of France?",
    "Who won the FIFA World Cup 2022?",
    "What is quantum computing?"
]

for q in test_questions:
    print(f"\nQuestion: {q}")
    print(rag_chain.invoke(q))
    print("="*60)

Zyro Dynamics Pvt. Ltd.
Company Profile
Doc Code: ZDL-CORP-001
Confidential — For Internal Use Only
Page 1
Zyro Dynamics Pvt. Ltd.
Navigate the Future
Company Profile
Document Code
ZDL-CORP-001
Version
V.01
Effective Date
01 April 2025
Document Owner
Corporate Communications
COMPANY OVERVIEW
Zyro Dynamics Pvt. Ltd. is a product-based enterprise software company headquartered in the United States. 
Founded in 2014 by Lamine Yamal and Aitana Bonmati, the company was built on a
single conviction: t


In [ ]:
import os

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Production-RAG-Assistant"